# KNN seguindo o exemplo da aula

Construímos um KNN do zero para classificar as amostras em controle (0) ou diabetes (1). Seguimos o exemplo da aula 5, página 12 do PDF, com normalização pelo mínimo e máximo, distância Euclidiana e votação pela classe mais frequente.

Comparamos alguns valores de K com validação cruzada, apresentada na aula 7. As referências aparecem ao longo do notebook. As páginas citadas são as posições no PDF, contando a capa.

Usamos a biblioteca padrão do Python para ler o CSV, organizar as listas e fazer as contas.

### Leitura da base padronizada

Lemos o arquivo produzido no notebook de exploração. Cada linha representa uma amostra, e cada coluna guarda um atributo. A coluna class é a resposta conhecida para o treinamento.

Usamos DictReader para acessar os valores pelos nomes das colunas e Counter para contar quantas amostras existem em cada classe.

In [1]:
from pathlib import Path
from collections import Counter
from statistics import median, mean
import csv
import math
import random

raiz = Path.cwd()
if not (raiz / "dados" / "treinamento_padronizado.csv").is_file():
    raiz = raiz.parent

caminho = raiz / "dados" / "treinamento_padronizado.csv"
with caminho.open(encoding="utf-8-sig", newline="") as arquivo:
    amostras = list(csv.DictReader(arquivo))

if not amostras:
    raise ValueError("A base de treinamento está vazia.")
if any(amostra["class"] not in {"0", "1"} for amostra in amostras):
    raise ValueError("Esperávamos as classes 0 e 1.")

pacientes = [amostra["PATIENT_CODE"].strip() for amostra in amostras]
if any(not paciente for paciente in pacientes) or len(set(pacientes)) != len(pacientes):
    raise ValueError("Conferir os códigos de paciente antes de dividir a base.")

print("Total de amostras:", len(amostras))
print("Amostras por classe:", dict(Counter(amostra["class"] for amostra in amostras)))

Total de amostras: 520
Amostras por classe: {'1': 270, '0': 250}


### Atributos desta versão

Começamos com as nove medidas numéricas que já exploramos. Elas permitem aplicar diretamente a fórmula de distância mostrada na aula 5. A escolha dessas colunas é nossa decisão para este experimento. O professor apresenta seleção de atributos na aula 2, páginas 78 a 81.

| Atributo | Informação |
|---|---|
| AGE | Idade |
| HEIGHT | Altura em centímetros |
| WEIGHT | Peso, sob a premissa de quilogramas |
| SYSTOLIC | Pressão sistólica |
| DIASTOLIC | Pressão diastólica |
| GLUCOSE | Glicose |
| HEMOGLOBIN | Hemoglobina, sob a premissa anterior de HbA1c |
| TRIGLYCERIDES | Triglicerídeos |
| SODIUM | Sódio |

As premissas sobre as medidas vieram da análise da base e continuam registradas como decisões do trabalho. Nesta versão, cada atributo entra com peso igual na fórmula da distância.

In [2]:
atributos = [
    "AGE", "HEIGHT", "WEIGHT", "SYSTOLIC", "DIASTOLIC",
    "GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES", "SODIUM",
]

print("Quantidade de atributos:", len(atributos))
print("Atributos:", ", ".join(atributos))

Quantidade de atributos: 9
Atributos: AGE, HEIGHT, WEIGHT, SYSTOLIC, DIASTOLIC, GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES, SODIUM


### Separação entre treinamento e validação

Reservamos 20% de cada classe para a avaliação local. Usamos a mesma semente 42 e a mesma divisão dos notebooks anteriores, com 416 amostras para escolher K e 104 para a comparação ao final.

A semente permite repetir o embaralhamento. A divisão por classe mantém aproximadamente a proporção entre controle e diabetes. As 104 amostras já foram consultadas em outros experimentos do projeto, então esse resultado precisa ser lido como uma comparação local.

O método holdout aparece na aula 7, página 7. A semente e a proporção de 80% e 20% são escolhas de implementação deste trabalho.

In [3]:
controle = [amostra for amostra in amostras if amostra["class"] == "0"]
diabetes = [amostra for amostra in amostras if amostra["class"] == "1"]

gerador = random.Random(42)
gerador.shuffle(controle)
gerador.shuffle(diabetes)

quantidade_controle = round(len(controle) * 0.20)
quantidade_diabetes = round(len(diabetes) * 0.20)

validacao = controle[:quantidade_controle] + diabetes[:quantidade_diabetes]
treinamento = controle[quantidade_controle:] + diabetes[quantidade_diabetes:]

print("Treinamento:", len(treinamento), dict(Counter(a["class"] for a in treinamento)))
print("Validação:", len(validacao), dict(Counter(a["class"] for a in validacao)))

Treinamento: 416 {'0': 200, '1': 216}
Validação: 104 {'0': 50, '1': 54}


### Valores ausentes

A aula 2, páginas 44 a 47 e 52, apresenta estimar os valores como uma das opções para lidar com ausências. Aqui escolhemos a mediana dos valores disponíveis no treinamento. A mediana é apresentada na aula 4, página 9. Essa forma específica de estimar é uma decisão nossa.

Mantemos o tratamento numérico que já vínhamos usando. Campos vazios, números não finitos e valores não positivos são tratados como ausentes. Também mantemos HEMOGLOBIN acima de 100 como ausente sob a premissa de HbA1c em porcentagem.

Por exemplo, se a mediana das idades no treinamento for 57, uma idade ausente recebe 57. Calculamos essa mediana novamente dentro de cada divisão, antes de avaliar as amostras que ficaram de fora.

In [4]:
def ler_numero(amostra, atributo):
    texto = str(amostra[atributo]).strip().replace(",", ".")
    try:
        valor = float(texto)
    except ValueError:
        return None

    if not math.isfinite(valor) or valor <= 0:
        return None
    if atributo == "HEMOGLOBIN" and valor > 100:
        return None
    return valor

### Normalização pelo mínimo e máximo

A fórmula apresentada na aula 5, página 12, é:

**valor normalizado = (valor − mínimo) / (máximo − mínimo)**

Se o mínimo de um atributo no treinamento for 50 e o máximo for 150, o valor 100 vira 0,5. O mínimo vira 0 e o máximo vira 1. Isso coloca as medidas em escalas comparáveis.

Calculamos o mínimo, o máximo e a mediana com as amostras de treinamento de cada divisão. Aplicamos esses mesmos parâmetros às amostras de avaliação. Se uma medida nova estiver fora dos extremos do treinamento, seu resultado pode ficar abaixo de 0 ou acima de 1.

Se todos os valores disponíveis de um atributo forem iguais, a diferença entre máximo e mínimo será zero. Nesse caso, atribuímos zero à coluna normalizada naquela divisão. Essa é uma escolha de implementação para tratar uma coluna que não varia na referência.

In [5]:
def calcular_parametros(amostras_referencia):
    parametros = {}

    for atributo in atributos:
        valores = [ler_numero(amostra, atributo) for amostra in amostras_referencia]
        disponiveis = [valor for valor in valores if valor is not None]
        if not disponiveis:
            raise ValueError(f"Nenhum valor disponível em {atributo} no treinamento.")

        parametros[atributo] = {
            "mediana": median(disponiveis),
            "minimo": min(disponiveis),
            "maximo": max(disponiveis),
        }

    return parametros


def normalizar_amostra(amostra, parametros):
    valores_normalizados = []

    for atributo in atributos:
        valor = ler_numero(amostra, atributo)
        parametro = parametros[atributo]
        if valor is None:
            valor = parametro["mediana"]

        amplitude = parametro["maximo"] - parametro["minimo"]
        if amplitude == 0:
            normalizado = 0.0
        else:
            normalizado = (valor - parametro["minimo"]) / amplitude

        valores_normalizados.append(normalizado)

    return valores_normalizados


parametros_treinamento = calcular_parametros(treinamento)

print("Atributo         | Mediana | Mínimo | Máximo")
for atributo, parametro in parametros_treinamento.items():
    print(
        f"{atributo:16} | {parametro['mediana']:>7.2f} | "
        f"{parametro['minimo']:>6.2f} | {parametro['maximo']:>6.2f}"
    )

Atributo         | Mediana | Mínimo | Máximo
AGE              |   57.00 |  16.00 |  98.00
HEIGHT           |  162.00 | 143.00 | 186.00
WEIGHT           |   76.40 |  41.30 | 242.10
SYSTOLIC         |  123.00 |  90.00 | 175.00
DIASTOLIC        |   81.00 |  60.00 | 107.00
GLUCOSE          |  103.00 |   3.10 | 500.00
HEMOGLOBIN       |    6.40 |   4.70 |  88.00
TRIGLYCERIDES    |  151.00 |  21.00 | 2880.00
SODIUM           |  140.00 | 133.00 | 149.00


### Distância Euclidiana

Para cada atributo normalizado, subtraímos o valor de uma amostra do valor da outra e elevamos a diferença ao quadrado. Somamos os quadrados e tiramos a raiz quadrada. Quanto menor o resultado, mais perto as duas amostras ficam nessa comparação.

A fórmula aparece na aula 3, página 13, e a aplicação no KNN aparece na aula 5, página 12. Por exemplo, duas diferenças de 0,3 e 0,4 dão uma distância de 0,5, pois a raiz de 0,3² + 0,4² é 0,5.

In [6]:
def distancia_euclidiana(primeira, segunda):
    soma = 0.0
    for valor_1, valor_2 in zip(primeira, segunda, strict=True):
        soma += (valor_1 - valor_2) ** 2
    return math.sqrt(soma)

### Votação dos vizinhos

Guardamos as medidas normalizadas e a classe de cada amostra do treinamento. Para classificar uma nova amostra, calculamos sua distância até todas as amostras guardadas e ordenamos da menor para a maior.

Cada um dos K primeiros vizinhos dá um voto. Com K igual a 3, se as classes forem 1, 0 e 1, a previsão será 1. Essa é a regra da classe mais frequente da aula 5, páginas 11 a 13.

Testamos apenas K ímpar. Como temos duas classes, sempre há uma maioria na votação. Para distâncias iguais, preservamos uma ordem de referência embaralhada com semente 42. Assim, a ordem inicial das classes na lista não decide automaticamente qual vizinho entra na última vaga.

In [7]:
def montar_referencia(amostras_referencia, parametros):
    referencia = []
    for amostra in amostras_referencia:
        numeros = normalizar_amostra(amostra, parametros)
        referencia.append((numeros, amostra["class"]))

    random.Random(42).shuffle(referencia)
    return referencia


def ordenar_vizinhos(amostra, referencia, parametros):
    numeros = normalizar_amostra(amostra, parametros)
    vizinhos = []

    for numeros_referencia, classe in referencia:
        distancia = distancia_euclidiana(numeros, numeros_referencia)
        vizinhos.append((distancia, classe))

    vizinhos.sort(key=lambda vizinho: vizinho[0])
    return vizinhos


def votar(vizinhos, k):
    if k < 1 or k % 2 == 0 or k > len(vizinhos):
        raise ValueError("K deve ser positivo, ímpar e caber na referência.")

    escolhidos = vizinhos[:k]
    votos = Counter(classe for distancia, classe in escolhidos)
    return votos.most_common(1)[0][0]

### Métricas para comparar os resultados

A aula 7, páginas 10 e 11, apresenta matriz de confusão, acurácia, precisão, recall e medida F.

- **Acurácia** é a proporção de amostras classificadas corretamente.
- **Precisão de uma classe** é a proporção de acertos entre as previsões daquela classe.
- **Recall de uma classe** é a proporção de acertos entre as amostras que realmente pertencem àquela classe.
- **F1** combina precisão e recall. Para o **F1 macro**, calculamos o F1 de cada classe e tiramos a média, atendendo à avaliação da competição.

Na matriz, as linhas são as classes reais e as colunas são as previsões. Mantemos a ordem 0 e 1. Se não houver previsões de uma classe, registramos sua precisão como zero.

In [8]:
def calcular_metricas(reais, previstas):
    if not reais or len(reais) != len(previstas):
        raise ValueError("As listas de classes devem ter o mesmo tamanho e estar preenchidas.")

    pares = Counter(zip(reais, previstas, strict=True))
    resultado = {
        "acuracia": sum(real == prevista for real, prevista in zip(reais, previstas)) / len(reais)
    }
    f1_por_classe = []

    for classe in ["0", "1"]:
        acertos = pares[classe, classe]
        total_real = sum(pares[classe, prevista] for prevista in ["0", "1"])
        total_previsto = sum(pares[real, classe] for real in ["0", "1"])

        precisao = acertos / total_previsto if total_previsto else 0.0
        recall = acertos / total_real if total_real else 0.0
        f1 = 2 * precisao * recall / (precisao + recall) if precisao + recall else 0.0

        resultado[f"precisao_{classe}"] = precisao
        resultado[f"recall_{classe}"] = recall
        f1_por_classe.append(f1)

    resultado["precisao_macro"] = (resultado["precisao_0"] + resultado["precisao_1"]) / 2
    resultado["f1_macro"] = mean(f1_por_classe)
    resultado["matriz"] = [
        [pares[real, prevista] for prevista in ["0", "1"]]
        for real in ["0", "1"]
    ]
    return resultado

### Validação cruzada em cinco partes

Dividimos as 416 amostras de treinamento em cinco partes. A cada rodada, uma parte é usada para avaliação e as outras quatro viram a referência. Cada amostra fica de fora uma vez, como no procedimento da aula 7, página 8.

Distribuímos as duas classes entre as partes. Com 416 amostras, uma parte terá 84 e as outras terão 83. Essa pequena diferença permite usar todas as amostras.

Em cada rodada, calculamos de novo as medianas, os mínimos e os máximos com as quatro partes de treinamento. Assim, os valores da parte avaliada não influenciam a preparação da referência.

In [9]:
partes = [[] for _ in range(5)]
gerador_partes = random.Random(42)

for classe in ["0", "1"]:
    grupo = [amostra for amostra in treinamento if amostra["class"] == classe]
    gerador_partes.shuffle(grupo)
    for indice, amostra in enumerate(grupo):
        partes[indice % 5].append(amostra)

for numero, parte in enumerate(partes, start=1):
    print(
        f"Parte {numero}: {len(parte)} amostras | "
        f"Classes: {dict(Counter(a['class'] for a in parte))}"
    )

Parte 1: 84 amostras | Classes: {'0': 40, '1': 44}
Parte 2: 83 amostras | Classes: {'0': 40, '1': 43}
Parte 3: 83 amostras | Classes: {'0': 40, '1': 43}
Parte 4: 83 amostras | Classes: {'0': 40, '1': 43}
Parte 5: 83 amostras | Classes: {'0': 40, '1': 43}


#### Comparação dos valores de K

Testamos K igual a 1, 3, 5, 7, 9 e 11. Para cada amostra da parte avaliada, ordenamos os vizinhos uma vez e contamos os votos para cada K.

Escolhemos o K com maior F1 macro médio nas cinco rodadas. Em caso de empate exato, ficamos com o menor K. Essa regra de escolha é nossa decisão para atender à métrica da competição.

In [10]:
valores_k = [1, 3, 5, 7, 9, 11]
metricas_por_k = {k: [] for k in valores_k}
reais_cruzada = []
previstas_cruzada = {k: [] for k in valores_k}

for indice, amostras_avaliacao in enumerate(partes):
    amostras_referencia = []
    for outro_indice, parte in enumerate(partes):
        if outro_indice != indice:
            amostras_referencia.extend(parte)

    parametros_rodada = calcular_parametros(amostras_referencia)
    referencia = montar_referencia(amostras_referencia, parametros_rodada)
    reais = [amostra["class"] for amostra in amostras_avaliacao]
    previstas_por_k = {k: [] for k in valores_k}

    for amostra in amostras_avaliacao:
        vizinhos = ordenar_vizinhos(amostra, referencia, parametros_rodada)
        for k in valores_k:
            previstas_por_k[k].append(votar(vizinhos, k))

    reais_cruzada.extend(reais)
    for k in valores_k:
        metricas_por_k[k].append(calcular_metricas(reais, previstas_por_k[k]))
        previstas_cruzada[k].extend(previstas_por_k[k])

    print(
        f"Rodada {indice + 1}: {len(amostras_referencia)} amostras de referência "
        f"e {len(amostras_avaliacao)} de avaliação"
    )

Rodada 1: 332 amostras de referência e 84 de avaliação
Rodada 2: 333 amostras de referência e 83 de avaliação
Rodada 3: 333 amostras de referência e 83 de avaliação
Rodada 4: 333 amostras de referência e 83 de avaliação
Rodada 5: 333 amostras de referência e 83 de avaliação


In [11]:
medias_por_k = {}

print(" K | Acurácia média | F1 macro médio | Recall controle | Recall diabetes")
for k, rodadas in metricas_por_k.items():
    medias = {}
    for metrica in ["acuracia", "f1_macro", "recall_0", "recall_1"]:
        medias[metrica] = mean(rodada[metrica] for rodada in rodadas)
    medias_por_k[k] = medias
    print(
        f"{k:>2} | {medias['acuracia']:>14.2%} | {medias['f1_macro']:>14.2%} | "
        f"{medias['recall_0']:>15.2%} | {medias['recall_1']:>15.2%}"
    )

melhor_k = min(valores_k, key=lambda k: (-medias_por_k[k]["f1_macro"], k))
print()
print(f"K escolhido: {melhor_k}")

 K | Acurácia média | F1 macro médio | Recall controle | Recall diabetes
 1 |         69.46% |         69.40% |          71.00% |          68.01%
 3 |         69.69% |         69.63% |          70.00% |          69.39%
 5 |         69.93% |         69.85% |          69.00% |          70.77%
 7 |         70.18% |         70.13% |          71.50% |          68.95%
 9 |         72.10% |         72.00% |          70.50% |          73.56%
11 |         72.34% |         72.24% |          70.00% |          74.48%

K escolhido: 11


#### Resultado das previsões fora da referência

Além da média por rodada, reunimos as 416 previsões feitas quando cada amostra ficou de fora. A acurácia desse conjunto é a quantidade total de acertos dividida por 416, como apresentado na aula 7, página 8.

O F1 calculado com todas as previsões juntas pode ser diferente da média dos cinco F1, porque as duas contas juntam os resultados de formas diferentes. Usamos a média por rodada para escolher K e mostramos as duas medidas com seus nomes.

In [12]:
metricas_cruzada = calcular_metricas(reais_cruzada, previstas_cruzada[melhor_k])
print(f"Acurácia nas 416 previsões: {metricas_cruzada['acuracia']:.2%}")
print(f"F1 macro nas 416 previsões: {metricas_cruzada['f1_macro']:.2%}")
print(f"F1 macro médio por rodada: {medias_por_k[melhor_k]['f1_macro']:.2%}")

Acurácia nas 416 previsões: 72.36%
F1 macro nas 416 previsões: 72.29%
F1 macro médio por rodada: 72.24%


### Avaliação nas 104 amostras reservadas

Fixamos o K escolhido e usamos as 416 amostras de treinamento como referência. Cada amostra reservada recebe uma previsão a partir das medidas normalizadas com os parâmetros desse treinamento.

O resultado das cinco rodadas serviu para escolher K. Esta avaliação permite acompanhar o comportamento da configuração nas mesmas amostras usadas como referência local nos outros notebooks.

In [13]:
referencia_final = montar_referencia(treinamento, parametros_treinamento)
reais_validacao = [amostra["class"] for amostra in validacao]
previstas_validacao = []

for amostra in validacao:
    vizinhos = ordenar_vizinhos(amostra, referencia_final, parametros_treinamento)
    previstas_validacao.append(votar(vizinhos, melhor_k))

metricas_validacao = calcular_metricas(reais_validacao, previstas_validacao)
matriz = metricas_validacao["matriz"]
acertos = matriz[0][0] + matriz[1][1]

print(f"K = {melhor_k} | Acertos: {acertos} de {len(validacao)}")
print(f"Acurácia: {metricas_validacao['acuracia']:.2%}")
print(f"F1 macro: {metricas_validacao['f1_macro']:.2%}")
print(f"Precisão macro: {metricas_validacao['precisao_macro']:.2%}")

for indice, classe in enumerate(["0", "1"]):
    nome = "controle" if classe == "0" else "diabetes"
    print(
        f"Classe {classe} ({nome}) | Precisão: {metricas_validacao[f'precisao_{classe}']:.2%} | "
        f"Recall: {metricas_validacao[f'recall_{classe}']:.2%} | Total real: {sum(matriz[indice])}"
    )

print()
print("            Previsto 0  Previsto 1")
print(f"Real 0      {matriz[0][0]:>10}  {matriz[0][1]:>10}")
print(f"Real 1      {matriz[1][0]:>10}  {matriz[1][1]:>10}")

K = 11 | Acertos: 67 de 104
Acurácia: 64.42%
F1 macro: 64.02%
Precisão macro: 64.53%
Classe 0 (controle) | Precisão: 65.12% | Recall: 56.00% | Total real: 50
Classe 1 (diabetes) | Precisão: 63.93% | Recall: 72.22% | Total real: 54

            Previsto 0  Previsto 1
Real 0              28          22
Real 1              15          39


#### Leitura desta execução

O K escolhido foi 11, com F1 macro médio de 72.24% nas cinco rodadas. Nas 104 amostras reservadas, acertamos 67, com acurácia de 64.42% e F1 macro de 64.02%.

Na matriz, 22 amostras de controle foram classificadas como diabetes e 15 amostras de diabetes foram classificadas como controle. Esse resultado pertence à configuração numérica descrita neste notebook e permite compará-la com as próximas versões fundamentadas nas aulas.